# 02 — Penrose Diagrams

**Spacetime Lab — Phase 2 concept + implementation notebook**

This notebook is the companion to `spacetime_lab.diagrams`. It walks through the physics of conformal compactification, shows how the Minkowski Penrose diagram is built from scratch, and then draws the full four-region Penrose diagram of the maximally extended Schwarzschild spacetime.

**What you will learn**

1. Why conformal transformations preserve causal structure
2. How the $\arctan$ compactification maps infinite Minkowski onto a finite diamond
3. The five kinds of infinity — $i^\pm$, $i^0$, $\mathscr{I}^\pm$ — and where they live
4. Eddington–Finkelstein → Kruskal → compactified null coordinates for Schwarzschild
5. Why the Schwarzschild singularity shows up as a *straight horizontal line* in the Penrose diagram (hint: $U + V = \pi/2$)
6. How to overlay world lines and light cones using the Spacetime Lab API

**References**

- Hawking & Ellis, *The Large Scale Structure of Space-Time*, ch. 5
- Carroll, *Spacetime and Geometry*, ch. 5 and appendix H
- Wald, *General Relativity*, ch. 11 and appendix D

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.diagrams import MinkowskiPenrose, SchwarzschildPenrose
from spacetime_lab.diagrams.render import render_matplotlib

plt.rcParams['figure.figsize'] = (6.5, 6.5)

## 1. Conformal transformations preserve causal structure

A conformal transformation is a pointwise rescaling of the metric:

$$
\tilde g_{\mu\nu}(x) = \Omega^2(x)\, g_{\mu\nu}(x), \qquad \Omega(x) > 0.
$$

**The key fact**: a vector $v^\mu$ is timelike / null / spacelike under $\tilde g$ iff it is timelike / null / spacelike under $g$. Proof: $\tilde g(v, v) = \Omega^2 g(v, v)$ has the same sign as $g(v, v)$ since $\Omega^2 > 0$.

This is why Penrose diagrams work. We can rescale the metric by a conformal factor that vanishes at infinity — pulling infinity to a finite coordinate — without disturbing which events can causally influence which. Light cones stay at the same angle.

## 2. Minkowski: the reference diagram

For 2D Minkowski $ds^2 = -dt^2 + dx^2$ introduce null coordinates $u = t - x$, $v = t + x$; then $ds^2 = -du\, dv$.

Compactify with $U = \arctan u$, $V = \arctan v$, both now in $(-\pi/2, \pi/2)$. Rescaling by $\Omega^2 = \cos^2 U \cos^2 V$ gives the flat metric $-dU\, dV$ on a finite square. After rotating to the familiar $(T, X) = ((U+V)/\sqrt{2}, (V-U)/\sqrt{2})$ orientation, we get a diamond:

In [ ]:
mink = MinkowskiPenrose()
print(mink.name, '—', mink.regions, 'region(s)')
print('coordinates:', mink.physical_coordinate_names)

# Mapping the origin and a few other events
for t, x in [(0, 0), (1, 0), (0, 2), (-2, 0)]:
    v = mink.physical_to_compact(1, t=t, x=x)
    print(f'  (t={t:+}, x={x:+}) -> (U={v.U:+.4f}, V={v.V:+.4f})')

In [ ]:
# Large |t| and |x| limits
for label, t, x in [
    ('i+ (t -> +inf)',  1e6,  0),
    ('i- (t -> -inf)', -1e6,  0),
    ('i0 right (x -> +inf)',   0,  1e6),
    ('i0 left  (x -> -inf)',   0, -1e6),
]:
    v = mink.physical_to_compact(1, t=t, x=x)
    print(f'{label:30s} -> (U={v.U:+.4f}, V={v.V:+.4f})')

In [ ]:
# Round-trip: forward then inverse should recover the original (t, x)
for t, x in [(0.3, 0.7), (-2, 3), (5, -4)]:
    p = mink.physical_to_compact(1, t=t, x=x)
    back = mink.compact_to_physical(1, p)
    print(f'(t={t:+.2f}, x={x:+.2f}) -> (U={p.U:+.4f}, V={p.V:+.4f}) -> {back}')

### The diagram itself

Static observers at $x = \text{const}$ appear as *vertical-ish curves* that start at $i^-$ and end at $i^+$. Light rays are 45° straight lines. Let's draw a static observer at $x = 0$ together with its light cone at the origin:

In [ ]:
mink_scene = mink.to_scene(
    world_lines=[(1, [{'t': tt, 'x': 0.0} for tt in np.linspace(-5, 5, 50)])],
    light_cones=[(1, {'t': 0.0, 'x': 0.0})],
)

fig, ax = plt.subplots(figsize=(6, 6))
render_matplotlib(mink_scene, ax=ax)
plt.tight_layout()
plt.show()

Notice how the infinities are arranged: $i^+$ at the top, $i^-$ at the bottom, $i^0$ on both left and right (1+1 Minkowski has *two* spatial infinities because $x \in \mathbb{R}$), and $\mathscr{I}^\pm$ on each of the four diagonal edges. The central green 'X' is the light cone at $(t,x)=(0,0)$ — two future-pointing null rays along $+U$ and $+V$, plus their past-pointing mirrors.

## 3. Schwarzschild: from tortoise to compactified null coordinates

For Schwarzschild we cannot just $\arctan$ the Schwarzschild coordinates directly — the metric is singular at $r = 2M$. We first go to tortoise coordinate:

$$r^*(r) = r + 2M \ln\!\left|\frac{r}{2M} - 1\right|,$$

then Eddington–Finkelstein null coordinates $u = t - r^*$, $v = t + r^*$, and finally Kruskal null coordinates

$$U_K = \mp \sqrt{|r/(2M) - 1|}\,e^{r/(4M)} e^{-t/(4M)}, \qquad V_K = \pm \sqrt{|r/(2M)-1|}\,e^{r/(4M)} e^{+t/(4M)},$$

with the signs chosen by *region* (I: $(-,+)$, II: $(+,+)$, III: $(+,-)$, IV: $(-,-)$). Apply $\arctan$ once more to compactify: $U = \arctan U_K$, $V = \arctan V_K$.

The `SchwarzschildPenrose` class does all of this for you.

In [ ]:
sch = SchwarzschildPenrose(mass=1.0)
print(sch.name)
print('regions:', sch.regions)
print('coords :', sch.physical_coordinate_names)

# Forward map: ISCO (r=6M) at t=0 in Region I
v = sch.physical_to_compact(1, t=0, r=6.0)
print(f'\n(t=0, r=6M)  ->  U={v.U:+.4f}, V={v.V:+.4f}')
# Round trip
back = sch.compact_to_physical(1, v)
print(f'round trip: {back}')

### The singularity is a straight line

In Region II, the equation for the future singularity ($r = 0$) is $U_K V_K = 1$, i.e. $\tan U \cdot \tan V = 1$. For $U, V \in (0, \pi/2)$ this is **exactly equivalent to $U + V = \pi/2$** — a straight line in compactified coordinates. That is the deep reason the singularity is drawn as a horizontal segment in textbook diagrams (not just as a cartoon convention).

In [ ]:
# Sample the singularity locus by taking r -> 0 in Region II with various t
print('Approach to the future singularity (r -> 0) in Region II:')
for r in [0.5, 0.1, 0.01, 1e-4, 1e-8]:
    v = sch.physical_to_compact(2, t=0, r=r)
    print(f'  r = {r:g}   U + V = {v.U + v.V:.8f}   (target pi/2 = {math.pi / 2:.8f})')
print('\nSame limit in Region IV (past singularity):')
for r in [0.5, 0.1, 1e-4]:
    v = sch.physical_to_compact(4, t=0, r=r)
    print(f'  r = {r:g}   U + V = {v.U + v.V:.8f}   (target -pi/2 = {-math.pi / 2:.8f})')

### The four-region diagram

Here is the full maximally extended Schwarzschild Penrose diagram, with:

- Two solid diamonds (Region I on the right, Region III on the left) carrying $\mathscr{I}^\pm$, $i^\pm$, $i^0$.
- Two red dashed horizontal lines: the future singularity (top, bounding Region II) and the past singularity (bottom, bounding Region IV).
- Four gray dashed diagonals: the two future horizons and two past horizons, meeting at the bifurcation sphere at the origin.
- A static observer at $r = 6M$ (the ISCO) in Region I, together with a light cone at $(t=0, r=6M)$.

In [ ]:
sch_scene = sch.to_scene(
    world_lines=[(1, [{'t': tt, 'r': 6.0} for tt in np.linspace(-5, 5, 40)])],
    light_cones=[(1, {'t': 0.0, 'r': 6.0})],
)

fig, ax = plt.subplots(figsize=(8, 6))
render_matplotlib(sch_scene, ax=ax)
plt.tight_layout()
plt.show()

### Three radial observers

Observers at different constant-$r$ values trace hyperbolae through Region I. An observer near the horizon ($r = 2.1 M$) piles up close to the horizon null line; an observer at $r = 10 M$ is almost straight. All three start at $i^-$ and end at $i^+$.

In [ ]:
ts = np.linspace(-8, 8, 80)
sch_scene2 = sch.to_scene(
    world_lines=[
        (1, [{'t': tt, 'r': 2.1} for tt in ts]),
        (1, [{'t': tt, 'r': 4.0} for tt in ts]),
        (1, [{'t': tt, 'r': 10.0} for tt in ts]),
    ],
)

fig, ax = plt.subplots(figsize=(8, 6))
render_matplotlib(sch_scene2, ax=ax)
plt.tight_layout()
plt.show()

## 4. Phase 2 gate checks

Before we move on to Phase 3 (Kerr geodesics) the following must hold. The assertions below fail loudly if anything regresses.

In [ ]:
# Minkowski round trips
for t, x in [(0, 0), (1.2, -0.7), (5, -4)]:
    p = mink.physical_to_compact(1, t=t, x=x)
    back = mink.compact_to_physical(1, p)
    assert math.isclose(back['t'], t, abs_tol=1e-12)
    assert math.isclose(back['x'], x, abs_tol=1e-12)

# Schwarzschild round trips in all four regions
for region, t, r in [(1, 1.5, 4.0), (2, 0.3, 1.2), (3, -2.0, 5.0), (4, 0.5, 0.8)]:
    p = sch.physical_to_compact(region, t=t, r=r)
    back = sch.compact_to_physical(region, p)
    assert math.isclose(back['t'], t, abs_tol=1e-8)
    assert math.isclose(back['r'], r, abs_tol=1e-8)

# Singularity is a straight line: U + V -> pi/2 as r -> 0 in Region II
v = sch.physical_to_compact(2, t=0, r=1e-10)
assert math.isclose(v.U + v.V, math.pi / 2, abs_tol=1e-6)

# Scene has the expected structure for Schwarzschild
scene = sch.to_scene()
kinds = [p.kind for p in scene.paths]
assert kinds.count('boundary') == 4
assert kinds.count('horizon') == 4
assert kinds.count('singularity') == 2
assert len(scene.infinities) == 10

# Minkowski has 4 edges and 8 infinities
mink_scene_bare = mink.to_scene()
assert len(mink_scene_bare.paths) == 4
assert len(mink_scene_bare.infinities) == 8

print('ALL PHASE 2 GATE CHECKS PASSED')

---

## What's next — Phase 3 teaser

Phase 3 adds **Kerr geodesics**: the full rotating-black-hole metric in Boyer–Lindquist coordinates, Carter's constant as the fourth conserved quantity, the ergosphere / ISCO / photon sphere for spinning black holes, and a symplectic integrator for timelike and null geodesics. Penrose diagrams for Kerr are trickier because the axisymmetry means you have to pick a $\theta$-slice; Phase 3 will handle that.

The extra kinds of infinity you'll meet along the way include the *Cauchy horizon*, the *inner horizon*, and the *ring singularity* — the Penrose diagram machinery from Phase 2 extends directly to handle them.